# Demo

Simple demo using `warp_transform()` with a BioTuring logo.

- **Input**: `bioturing_input.png` (RGB image, H x W x 3)
- **Transform**: Affine rotation + scale
- **Output**: Warped image side-by-side with original

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

from spatialx_transform import warp_transform
from spatialx_transform.transforms import Affine
from spatialx_transform.params import AffineParams

import logging
from pathlib import Path


def setup_logging(
    name: str = "spatialx_transform",
    level: int = logging.DEBUG,
    log_dir: str = "logging",
) -> logging.Logger:
    """Configure file-only logging. Call once before using warp_transform.

    Creates a log file at <log_dir>/warp.log relative to the current
    working directory.
    """
    log_path = Path(log_dir)
    log_path.mkdir(exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(level)
    fh = logging.FileHandler(log_path / "warp.log")
    fh.setLevel(level)
    fh.setFormatter(
        logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s")
    )
    logger.addHandler(fh)
    return logger


setup_logging()

In [ ]:
# Load logo: cv2 reads as [H, W, 3] BGR
img_bgr = cv.imread("bioturing_input.png")
img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)

# Convert to [C, H, W] for warp_transform
img_chw = np.transpose(img_rgb, (2, 0, 1))
print(f"Input shape: {img_chw.shape}")

In [ ]:
# Define a simple affine transform: rotate 15deg + scale 1.2x
angle = 15 * np.pi / 180
s = 1.2
A = [[s * np.cos(angle), -s * np.sin(angle)], [s * np.sin(angle), s * np.cos(angle)]]
b = [0.0, 0.0]

tf = Affine(params=AffineParams(A=A, b=b))
print(f"A = {A}")
print(f"b = {b}")

In [ ]:
# Preflight
preflight_result = warp_transform(
    img_chw, tf, d=(10, 10), scale=(1.0, 1.0), preflight=True
)
preflight_result

In [ ]:
# Warp
result = warp_transform(img_chw, tf, d=(10, 10), scale=(1.0, 1.0), preflight=False)
print(f"Output shape: {result.img.shape}")
print(f"Offset: {result.offset}")

In [ ]:
# Convert back to HWC for display
output_rgb = np.transpose(result.img, (1, 2, 0))

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(img_rgb)
axes[0].set_title(f"Input ({img_rgb.shape[1]}x{img_rgb.shape[0]})")
axes[0].axis("off")
axes[1].imshow(output_rgb)
axes[1].set_title(f"Warped ({output_rgb.shape[1]}x{output_rgb.shape[0]})")
axes[1].axis("off")
plt.tight_layout()
plt.show()